In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

clean_path = PROCESSED_DIR / "clean_prices.csv"

clean_prices = pd.read_csv(clean_path, parse_dates=["date"])
clean_prices = clean_prices.sort_values("date").set_index("date")

clean_prices.head()

,Open_AAPL,High_AAPL,Low_AAPL,Close_AAPL,Adj_Close_AAPL,Volume_AAPL,Open_AMZN,High_AMZN,Low_AMZN,Close_AMZN,...,Low_VOO,Close_VOO,Adj_Close_VOO,Volume_VOO,Open_SPY,High_SPY,Low_SPY,Close_SPY,Adj_Close_SPY,Volume_SPY
date,,,,,,,,,,,,,,,,,,,,,
2021-02-01,133.750000,135.380005,130.929993,134.139999,130.578918,106239800,162.117996,167.513000,161.751495,167.143997,...,341.399994,345.769989,321.844574,3334000,373.720001,377.339996,370.380005,376.230011,351.201813,75817600
2021-02-02,135.729996,136.309998,134.610001,134.990005,131.406326,83305400,169.000000,171.386993,168.056503,169.000000,...,349.000000,350.739990,326.470734,2955200,379.649994,383.220001,376.320007,381.549988,356.167877,64450700
2021-02-03,135.759995,135.770004,133.610001,133.940002,130.384186,89880900,171.250504,171.699997,165.431000,165.626495,...,349.880005,351.089996,326.796509,3392800,382.440002,383.700012,380.480011,381.850006,356.447937,52427100
2021-02-04,136.300003,137.399994,134.589996,137.389999,133.742615,84183100,166.500000,167.350006,163.887497,166.550003,...,351.920013,355.019989,330.454498,1988300,382.959991,386.239990,381.970001,386.190002,360.499207,47142600
2021-02-05,137.350006,137.419998,135.860001,136.759995,133.328278,75693800,165.949997,168.850006,165.135498,167.607498,...,355.320007,356.440002,331.776184,2091800,388.200012,388.470001,386.140015,387.709991,361.918152,48669800


In [14]:
# Select adjusted close columns like Adj_Close_AAPL, Adj_Close_AMZN, ...
price_cols = [c for c in clean_prices.columns if c.startswith("Adj_Close_")]
price_cols

['Adj_Close_AAPL',
 'Adj_Close_AMZN',
 'Adj_Close_MSFT',
 'Adj_Close_VOO',
 'Adj_Close_SPY']

In [15]:
# Ensure numeric
clean_prices[price_cols] = clean_prices[price_cols].apply(pd.to_numeric, errors="coerce")

# Daily simple returns
returns = clean_prices[price_cols].pct_change()
returns.columns = [c.replace("Adj_Close_", "ret_") for c in returns.columns]

# Optionally also create log returns (not strictly required for lags, but nice to have)
log_returns = np.log(clean_prices[price_cols] / clean_prices[price_cols].shift(1))
log_returns.columns = [c.replace("Adj_Close_", "logret_") for c in log_returns.columns]

# Base feature DF (prices + returns)
feat_df = clean_prices.join(returns).join(log_returns)

feat_df.head()

,Open_AAPL,High_AAPL,Low_AAPL,Close_AAPL,Adj_Close_AAPL,Volume_AAPL,Open_AMZN,High_AMZN,Low_AMZN,Close_AMZN,...,ret_AAPL,ret_AMZN,ret_MSFT,ret_VOO,ret_SPY,logret_AAPL,logret_AMZN,logret_MSFT,logret_VOO,logret_SPY
date,,,,,,,,,,,,,,,,,,,,,
2021-02-01,133.750000,135.380005,130.929993,134.139999,130.578918,106239800,162.117996,167.513000,161.751495,167.143997,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-02,135.729996,136.309998,134.610001,134.990005,131.406326,83305400,169.000000,171.386993,168.056503,169.000000,...,0.006336,0.011104,-0.000584,0.014374,0.014140,0.006316,0.011043,-0.000584,0.014272,0.014041
2021-02-03,135.759995,135.770004,133.610001,133.940002,130.384186,89880900,171.250504,171.699997,165.431000,165.626495,...,-0.007778,-0.019962,0.014571,0.000998,0.000786,-0.007809,-0.020163,0.014466,0.000997,0.000786
2021-02-04,136.300003,137.399994,134.589996,137.389999,133.742615,84183100,166.500000,167.350006,163.887497,166.550003,...,0.025758,0.005576,-0.004074,0.011193,0.011366,0.025432,0.005560,-0.004083,0.011131,0.011302
2021-02-05,137.350006,137.419998,135.860001,136.759995,133.328278,75693800,165.949997,168.850006,165.135498,167.607498,...,-0.003098,0.006349,0.000785,0.004000,0.003936,-0.003103,0.006329,0.000785,0.003992,0.003928


In [16]:
ret_cols = [c for c in feat_df.columns if c.startswith("ret_")]
lags = [1, 2, 5]

for col in ret_cols:
    for l in lags:
        feat_df[f"{col}_lag{l}"] = feat_df[col].shift(l)

feat_df.filter(regex="^ret_.*lag").head()

,ret_AAPL_lag1,ret_AAPL_lag2,ret_AAPL_lag5,ret_AMZN_lag1,ret_AMZN_lag2,ret_AMZN_lag5,ret_MSFT_lag1,ret_MSFT_lag2,ret_MSFT_lag5,ret_VOO_lag1,ret_VOO_lag2,ret_VOO_lag5,ret_SPY_lag1,ret_SPY_lag2,ret_SPY_lag5
date,,,,,,,,,,,,,,,
2021-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-03,0.006336,NaN,NaN,0.011104,NaN,NaN,-0.000584,NaN,NaN,0.014374,NaN,NaN,0.014140,NaN,NaN
2021-02-04,-0.007778,0.006336,NaN,-0.019962,0.011104,NaN,0.014571,-0.000584,NaN,0.000998,0.014374,NaN,0.000786,0.014140,NaN
2021-02-05,0.025758,-0.007778,NaN,0.005576,-0.019962,NaN,-0.004074,0.014571,NaN,0.011193,0.000998,NaN,0.011366,0.000786,NaN


In [17]:
# Adjusted close columns (same as before)
price_cols = [c for c in feat_df.columns if c.startswith("Adj_Close_")]

ma_windows = [5, 10]

for col in price_cols:
    ticker = col.replace("Adj_Close_", "")  # e.g. "AAPL"
    
    for w in ma_windows:
        ma_col = f"MA{w}_{ticker}"
        diff_col = f"price_minus_MA{w}_{ticker}"
        
        # Simple moving average over w days
        feat_df[ma_col] = feat_df[col].rolling(window=w).mean()
        
        # Price - moving average (how far above/below MA)
        feat_df[diff_col] = feat_df[col] - feat_df[ma_col]

In [28]:
feat_df.filter(regex="^MA[0-9]+_").head()
feat_df.filter(regex="^price_minus_MA").head(10)

,price_minus_MA5_AAPL,price_minus_MA10_AAPL,price_minus_MA5_AMZN,price_minus_MA10_AMZN,price_minus_MA5_MSFT,price_minus_MA10_MSFT,price_minus_MA5_VOO,price_minus_MA10_VOO,price_minus_MA5_SPY,price_minus_MA10_SPY
date,,,,,,,,,,
2021-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-05,1.440213,NaN,0.421899,NaN,0.888535,NaN,4.307684,NaN,4.671155,NaN
2021-02-08,1.007309,NaN,-0.839197,NaN,0.606427,NaN,4.279858,NaN,4.618854,NaN
2021-02-09,-0.108228,NaN,-0.986200,NaN,1.036307,NaN,2.513177,NaN,2.751892,NaN
2021-02-10,-1.034381,NaN,-1.647705,NaN,0.159317,NaN,0.977362,NaN,1.056647,NaN
2021-02-11,-0.887183,NaN,-2.181494,NaN,1.285770,NaN,0.735388,NaN,0.800916,NaN


In [19]:
# Daily return columns
ret_cols = [c for c in feat_df.columns if c.startswith("ret_")]

vol_windows = [10, 20]

for col in ret_cols:
    ticker = col.replace("ret_", "")  # e.g. "AAPL"
    
    for w in vol_windows:
        vol_col = f"vol{w}_{ticker}"
        feat_df[vol_col] = feat_df[col].rolling(window=w).std()

C:\Users\milli\AppData\Local\Temp\ipykernel_18212\4028976661.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feat_df[vol_col] = feat_df[col].rolling(window=w).std()


In [23]:
feat_df.filter(regex="^vol(10|20)_").head(20)

,vol10_AAPL,vol20_AAPL,vol10_AMZN,vol20_AMZN,vol10_MSFT,vol20_MSFT,vol10_VOO,vol20_VOO,vol10_SPY,vol20_SPY,...,vol10_VOO_lag2,vol20_VOO_lag2,vol10_VOO_lag5,vol20_VOO_lag5,vol10_SPY_lag1,vol20_SPY_lag1,vol10_SPY_lag2,vol20_SPY_lag2,vol10_SPY_lag5,vol20_SPY_lag5
date,,,,,,,,,,,,,,,,,,,,,
2021-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# Choose SPY as primary market proxy (you can add VOO similarly)
market_col = "ret_SPY"   # must match your ret_* naming

if market_col in feat_df.columns:
    for col in ret_cols:
        if col == market_col:
            continue  # skip SPY vs itself
        
        ticker = col.replace("ret_", "")  # e.g. "AAPL"
        diff_col = f"ret_diff_SPY_{ticker}"
        
        # Stock's daily return relative to SPY's daily return
        feat_df[diff_col] = feat_df[col] - feat_df[market_col]

C:\Users\milli\AppData\Local\Temp\ipykernel_18212\1071356924.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feat_df[diff_col] = feat_df[col] - feat_df[market_col]


In [ ]:
feat_df.filter(regex="^ret_diff_SPY_").head(20)

,ret_diff_SPY_AAPL,ret_diff_SPY_AMZN,ret_diff_SPY_MSFT,ret_diff_SPY_VOO,ret_diff_SPY_AAPL_lag1,ret_diff_SPY_AAPL_lag2,ret_diff_SPY_AAPL_lag5,ret_diff_SPY_AMZN_lag1,ret_diff_SPY_AMZN_lag2,ret_diff_SPY_AMZN_lag5,ret_diff_SPY_MSFT_lag1,ret_diff_SPY_MSFT_lag2,ret_diff_SPY_MSFT_lag5,ret_diff_SPY_VOO_lag1,ret_diff_SPY_VOO_lag2,ret_diff_SPY_VOO_lag5,ret_diff_SPY_SPY_lag1,ret_diff_SPY_SPY_lag2,ret_diff_SPY_SPY_lag5
date,,,,,,,,,,,,,,,,,,,
2021-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-02,-0.007804,-0.003036,-0.014724,0.000234,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02-03,-0.008565,-0.020748,0.013785,0.000212,0.005550,NaN,NaN,0.010318,NaN,NaN,-0.001370,NaN,NaN,0.013588,NaN,NaN,0.013354,NaN,NaN
2021-02-04,0.014392,-0.005790,-0.015440,-0.000172,-0.019144,-0.005029,NaN,-0.031327,-0.000261,NaN,0.003206,-0.011950,NaN,-0.010368,0.003008,NaN,-0.010579,0.002775,NaN
2021-02-05,-0.007034,0.002413,-0.003151,0.000064,0.021822,-0.011715,NaN,0.001640,-0.023898,NaN,-0.008010,0.010635,NaN,0.007257,-0.002938,NaN,0.007430,-0.003150,NaN


In [29]:
# We'll define a separate target per ticker, e.g., up_next_day_AAPL
ret_cols = [c for c in feat_df.columns if c.startswith("ret_")]

for col in ret_cols:
    ticker = col.replace("ret_", "")  # e.g. "AAPL"
    target_col = f"up_next_day_{ticker}"
    
    # Next day's return for this ticker
    next_ret = feat_df[col].shift(-1)
    
    # 1 if next day's return > 0, else 0 (and NaN stays NaN)
    feat_df[target_col] = (next_ret > 0).astype("Int64")

C:\Users\milli\AppData\Local\Temp\ipykernel_18212\2959259085.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feat_df[target_col] = (next_ret > 0).astype("Int64")
C:\Users\milli\AppData\Local\Temp\ipykernel_18212\2959259085.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feat_df[target_col] = (next_ret > 0).astype("Int64")
C:\Users\milli\AppData\Local\Temp\ipykernel_18212\2959259085.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor

In [31]:
# Look at one ticker to verify
feat_df[["ret_AAPL", "up_next_day_AAPL"]].head(10)
feat_df[["ret_AAPL", "up_next_day_AAPL"]].tail(10)

,ret_AAPL,up_next_day_AAPL
date,,
2026-01-14,-0.004175,0
2026-01-15,-0.006732,0
2026-01-16,-0.010379,0
2026-01-20,-0.034556,1
2026-01-21,0.003851,1
2026-01-22,0.002827,0
2026-01-23,-0.001248,1
2026-01-26,0.029713,1
2026-01-27,0.011198,0


In [32]:
# Pick one ticker's target for this dataset (you can make others later)
target_col = "up_next_day_AAPL"

# Basic feature groups
price_cols      = [c for c in feat_df.columns if c.startswith("Adj_Close_")]
ret_cols        = [c for c in feat_df.columns if c.startswith("ret_")]
lag_cols        = [c for c in feat_df.columns if "_lag" in c and c.startswith("ret_")]
ma_cols         = [c for c in feat_df.columns if c.startswith("MA")]
price_ma_cols   = [c for c in feat_df.columns if c.startswith("price_minus_MA")]
vol_cols        = [c for c in feat_df.columns if c.startswith("vol")]
market_diff_cols= [c for c in feat_df.columns if c.startswith("ret_diff_SPY_")]

feature_cols = (
    ret_cols
    + lag_cols
    + ma_cols
    + price_ma_cols
    + vol_cols
    + market_diff_cols
)
len(feature_cols), target_col

(148, 'up_next_day_AAPL')

In [33]:
model_df = feat_df[feature_cols + [target_col]].copy()

# Drop rows where target is NA (last day) or features have NaNs from rolling windows
model_df = model_df.dropna(subset=[target_col])
model_df = model_df.dropna()  # or model_df.fillna(0) if you prefer

model_df.head(), model_df.shape

(            ret_AAPL  ret_AMZN  ret_MSFT   ret_VOO   ret_SPY  ret_AAPL_lag1  \
 date                                                                          
 2021-03-09  0.040650  0.037568  0.028102  0.013990  0.014277      -0.041674   
 2021-03-10 -0.009167 -0.001701 -0.005817  0.006182  0.006225       0.040650   
 2021-03-11  0.016503  0.018298  0.020265  0.010557  0.010139      -0.009167   
 2021-03-12 -0.007626 -0.007740 -0.005820  0.001409  0.001347       0.016503   
 2021-03-15  0.024457 -0.002528 -0.003987  0.005795  0.005963      -0.007626   
 
             ret_AAPL_lag2  ret_AAPL_lag5  ret_AMZN_lag1  ret_AMZN_lag2  ...  \
 date                                                                    ...   
 2021-03-09       0.010739      -0.020894      -0.016167       0.007687  ...   
 2021-03-10      -0.041674      -0.024457       0.037568      -0.016167  ...   
 2021-03-11       0.040650      -0.015812      -0.001701       0.037568  ...   
 2021-03-12      -0.009167       0.010

In [34]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
out_path = PROCESSED_DIR / "model_data.csv"

model_df.to_csv(out_path, index=True)  # index is the date
out_path

WindowsPath('../data/processed/model_data.csv')